<a href="https://colab.research.google.com/github/weagan/MAX-mojo/blob/main/cuda_vs_max_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUDA (PyTorch) vs Modular MAX — LLM inference benchmark

Runs on **Google Colab** and **Kaggle**. No Modular API key is needed: MAX is installed from PyPI and runs locally.

**Flow**
1. Load secrets (optional `HF_TOKEN` for Hugging Face downloads / gated models)
2. **Part 1:** run the model with PyTorch + CUDA and record latency / throughput
3. Free the GPU, install MAX in an isolated virtualenv, start `max serve` locally
4. **Part 2:** run the identical benchmark against MAX
5. Compare in a table and charts

**Secrets setup (optional but recommended)**
- Colab: key icon in the left sidebar, add `HF_TOKEN`, enable *Notebook access*
- Kaggle: *Add-ons → Secrets*, add `HF_TOKEN`, attach it to the notebook
- Kaggle also needs **Internet: On** and a GPU accelerator

**GPU notes:** MAX supports a limited set of GPUs. Ampere or newer (A100, L4, A10, RTX 30/40, H100) is the safest choice. T4 may work; Kaggle's P100 (Pascal) is not supported by MAX, so Part 2 will fail there.

In [1]:
import os, sys, subprocess

def get_secret(name, default=None):
    """Read a secret from Colab, Kaggle, or environment variables (in that order)."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception:
        pass
    return os.environ.get(name, default)

PLATFORM = "colab" if "google.colab" in sys.modules else ("kaggle" if os.path.exists("/kaggle") else "other")
HF_TOKEN = get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"Platform: {PLATFORM} | HF_TOKEN found: {bool(HF_TOKEN)}  (a token is only required for gated models)")

Platform: colab | HF_TOKEN found: True  (a token is only required for gated models)


In [2]:
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU - enable a GPU runtime!")

Sat Oct 17 18:18:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ---------------- Config ----------------
MODEL        = "Qwen/Qwen2.5-1.5B-Instruct"   # any model supported by BOTH transformers and MAX
                                              # e.g. "meta-llama/Llama-3.2-1B-Instruct" (gated: needs HF_TOKEN)
NEW_TOKENS   = 128      # tokens generated per request
BATCH        = 8        # concurrent requests for the throughput test
MAX_LEN      = 2048     # context length for MAX server
PORT         = 8000

PROMPTS = [
    "Explain how a GPU executes a matrix multiplication, step by step.",
    "Write a short story about a robot learning to paint.",
    "What are the main differences between CPUs and GPUs?",
    "Summarize the causes of the French Revolution.",
    "Give me a Python function that checks if a string is a palindrome, and explain it.",
]

import torch
cc = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
# bf16 needs Ampere+; fall back to fp32 on older GPUs so BOTH stacks use the same precision
DTYPE_NAME = "bfloat16" if cc[0] >= 8 else "float32"
TORCH_DTYPE = getattr(torch, DTYPE_NAME)
print(f"GPU: {GPU_NAME} (sm_{cc[0]}{cc[1]}) | precision for both stacks: {DTYPE_NAME}")

GPU: Tesla T4 (sm_75) | precision for both stacks: float32


In [4]:
import statistics as st, time, json, gc

def run_suite(name, gen_fn):
    """
    gen_fn(prompts: list[str], max_new: int) -> (wall_seconds, total_new_tokens)
    Identical benchmark for both backends.
    """
    print(f"\n=== {name} ===")
    for _ in range(2): gen_fn(PROMPTS[:1], 16)                      # warmup

    ttft = [gen_fn([p], 1)[0] * 1000 for p in PROMPTS]               # time to first token (ms)
    single = []
    for p in PROMPTS:
        dt, n = gen_fn([p], NEW_TOKENS)
        single.append(n / dt)
    batch_prompts = (PROMPTS * BATCH)[:BATCH]
    batch = []
    for _ in range(3):
        dt, n = gen_fn(batch_prompts, NEW_TOKENS)
        batch.append(n / dt)

    res = {
        "ttft_ms": st.median(ttft),
        "single_tok_s": st.median(single),
        f"batch{BATCH}_tok_s": st.median(batch),
    }
    for k, v in res.items(): print(f"{k:>20}: {v:,.1f}")
    return res

RESULTS = {}

## Part 1 — PyTorch + CUDA (Hugging Face Transformers)

In [5]:
%pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 37.1 MB/s eta 0:00:00


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings; warnings.filterwarnings("ignore")

tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=TORCH_DTYPE, token=HF_TOKEN).to("cuda").eval()

def cuda_gen(prompts, max_new):
    texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
    torch.cuda.synchronize(); t = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False)
    torch.cuda.synchronize(); dt = time.perf_counter() - t
    new = out[:, enc["input_ids"].shape[1]:]
    n = int((new != tok.pad_token_id).sum().item())
    return dt, max(n, 1)

RESULTS["PyTorch CUDA"] = run_suite("PyTorch + CUDA", cuda_gen)
RESULTS["PyTorch CUDA"]["gpu_mem_gb"] = torch.cuda.max_memory_allocated() / 1e9
print(f"peak GPU memory: {RESULTS['PyTorch CUDA']['gpu_mem_gb']:.2f} GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


=== PyTorch + CUDA ===
             ttft_ms: 68.5
        single_tok_s: 28.8
        batch8_tok_s: 143.0
peak GPU memory: 6.31 GB


In [7]:
# Free the GPU so MAX can have it
del model; gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory still allocated by torch: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory still allocated by torch: 0.01 GB


## Part 2 — Modular MAX (local, no API key)

MAX is installed into a **separate virtualenv** so it doesn't clash with the PyTorch packages already on Colab/Kaggle.

In [8]:
VENV = "/tmp/max_env"
if not os.path.exists(f"{VENV}/bin/max"):
    subprocess.run([sys.executable, "-m", "venv", VENV], check=True)
    r = subprocess.run([f"{VENV}/bin/pip", "install", "-q", "-U", "pip", "modular"], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
print(subprocess.run([f"{VENV}/bin/max", "--version"], capture_output=True, text=True).stdout)

CalledProcessError: Command '['/usr/bin/python3', '-m', 'venv', '/tmp/max_env']' returned non-zero exit status 1.

In [ ]:
import requests

LOG = open("/tmp/max_serve.log", "w")
cmd = [f"{VENV}/bin/max", "serve",
       "--model-path", MODEL,
       "--max-length", str(MAX_LEN),
       "--quantization-encoding", DTYPE_NAME,
       "--port", str(PORT)]
print(" ".join(cmd))
server = subprocess.Popen(cmd, stdout=LOG, stderr=subprocess.STDOUT, env=os.environ.copy())

BASE = f"http://127.0.0.1:{PORT}"
t0 = time.time()
while True:
    if server.poll() is not None:
        print(open("/tmp/max_serve.log").read()[-4000:])
        raise RuntimeError("max serve exited. See log above (unsupported GPU / flag change? try `max serve --help`).")
    try:
        if requests.get(f"{BASE}/v1/models", timeout=2).status_code == 200: break
    except Exception:
        pass
    if time.time() - t0 > 1200:
        print(open("/tmp/max_serve.log").read()[-4000:]); raise TimeoutError("MAX server did not start in 20 min")
    time.sleep(5)
print(f"MAX server ready after {time.time()-t0:.0f}s")

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def _one(prompt, max_new):
    r = requests.post(f"{BASE}/v1/chat/completions", timeout=600, json={
        "model": MODEL, "max_tokens": max_new, "temperature": 0,
        "messages": [{"role": "user", "content": prompt}]})
    r.raise_for_status()
    return r.json()["usage"]["completion_tokens"]

def max_gen(prompts, max_new):
    t = time.perf_counter()
    with ThreadPoolExecutor(len(prompts)) as ex:
        ns = list(ex.map(lambda p: _one(p, max_new), prompts))
    return time.perf_counter() - t, max(sum(ns), 1)

RESULTS["MAX"] = run_suite("Modular MAX", max_gen)

# MAX pre-allocates GPU memory, so report what the device shows as used
out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
RESULTS["MAX"]["gpu_mem_gb"] = float(out.strip().splitlines()[0]) / 1024

## Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(RESULTS).T
bk = f"batch{BATCH}_tok_s"
df = df[["ttft_ms", "single_tok_s", bk, "gpu_mem_gb"]].round(2)
display(df)

speed = {
    "TTFT speedup (x, lower latency is better)": df.loc["PyTorch CUDA","ttft_ms"] / df.loc["MAX","ttft_ms"],
    "Single-stream speedup (x)": df.loc["MAX","single_tok_s"] / df.loc["PyTorch CUDA","single_tok_s"],
    f"Batch-{BATCH} throughput speedup (x)": df.loc["MAX",bk] / df.loc["PyTorch CUDA",bk],
}
for k, v in speed.items(): print(f"{k}: {v:.2f}x")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, col, title in zip(ax, ["ttft_ms", "single_tok_s", bk],
                         ["Time to first token (ms) - lower is better", "Single request tokens/s", f"{BATCH} concurrent requests: total tokens/s"]):
    df[col].plot.bar(ax=a, color=["#76b900", "#ff5a1f"], rot=0)
    a.set_title(title, fontsize=10)
    for i, v in enumerate(df[col]): a.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
fig.suptitle(f"{MODEL} on {GPU_NAME} ({DTYPE_NAME})")
plt.tight_layout(); plt.show()

with open("benchmark_results.json", "w") as f:
    json.dump({"model": MODEL, "gpu": GPU_NAME, "dtype": DTYPE_NAME, "results": RESULTS}, f, indent=2)

In [ ]:
# Cleanup: stop the MAX server
server.terminate()
try: server.wait(timeout=30)
except Exception: server.kill()
print("MAX server stopped.")

### Notes on interpreting the results
- **Not a pure kernel comparison.** Part 1 is Hugging Face `generate()` (eager PyTorch on CUDA); Part 2 is the full MAX serving stack (graph compiler, continuous batching, paged KV cache). The gap reflects the whole stack, not just Mojo vs CUDA kernels. For a tougher CUDA baseline, swap Part 1 for vLLM or TensorRT-LLM.
- MAX pre-allocates GPU memory for its KV cache, so its memory number will look larger.
- If `max serve` flags have changed in your installed version, run `!{VENV}/bin/max serve --help`.
- Run the notebook more than once; first runs include download and compile time (excluded from the timings by the warmup).